# LayoutVLM 完整复现 - CVPR 2025

**论文**: [LayoutVLM: Differentiable Optimization of 3D Layout via Vision-Language Models](https://arxiv.org/abs/2412.02193)

**GitHub**: https://github.com/sunfanyunn/LayoutVLM

---

## 🎯 功能说明

此notebook整合了完整的工作流程：
1. ✅ 在Colab中安装Blender 4.2.1
2. ✅ 配置GPU渲染
3. ✅ 运行LayoutVLM生成3D布局
4. ✅ 自动渲染可视化结果

---

## 📋 使用步骤

1. **启用GPU**: 运行时 → 更改运行时类型 → T4 GPU
2. **高RAM（推荐）**: 运行时 → 更改运行时类型 → 高RAM
3. **按顺序执行**所有单元格
4. **配置API**: 在步骤8中填入你的API密钥

⏱️ **预计时间**: 首次运行约15-20分钟（后续约5-10分钟）

---

In [72]:
# 检查GPU
print('='*60)
print('🔍 检查GPU环境')
print('='*60)

!nvidia-smi

print('\n' + '='*60)
print('✅ GPU检查完成')
print('='*60)
print('\n⚠️  如果看不到GPU信息，请检查运行时设置')

🔍 检查GPU环境
Mon Nov  3 08:38:02 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             55W /  400W |       5MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-------------------------------------

In [73]:
from google.colab import drive
import os

# 挂载Drive
drive.mount('/content/drive')

# 创建项目目录结构
PROJECT_DIR = '/content/drive/MyDrive/LayoutVLM_Project'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/datasets', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/blender_scripts', exist_ok=True)

print('\n' + '='*60)
print('✅ Google Drive已挂载')
print('='*60)
print(f'📁 项目目录: {PROJECT_DIR}')
print(f'📊 结果目录: {PROJECT_DIR}/results')
print(f'💾 数据集目录: {PROJECT_DIR}/datasets')
print('='*60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

✅ Google Drive已挂载
📁 项目目录: /content/drive/MyDrive/LayoutVLM_Project
📊 结果目录: /content/drive/MyDrive/LayoutVLM_Project/results
💾 数据集目录: /content/drive/MyDrive/LayoutVLM_Project/datasets


In [74]:
import os

print('='*60)
print('🔧 安装Blender 4.2.1')
print('='*60)

# 1. 安装系统依赖
print('\n📦 步骤1/4: 安装系统依赖...')
!apt-get update -y -qq > /dev/null 2>&1
!apt-get install -y -qq xvfb libegl1-mesa libxrandr2 libxinerama1 libxxf86vm1 libxi6 wget > /dev/null 2>&1
print('   ✅ 系统依赖安装完成')

# 2. 下载Blender
print('\n📥 步骤2/4: 下载Blender 4.2.1...')
BLENDER_URL = "https://download.blender.org/release/Blender4.2/blender-4.2.1-linux-x64.tar.xz"
BLENDER_FILE = "blender-4.2.1-linux-x64.tar.xz"

if not os.path.exists(f'/content/{BLENDER_FILE}'):
    !wget -q --show-progress {BLENDER_URL}
    print('   ✅ 下载完成')
else:
    print('   ✅ Blender压缩包已存在')

# 3. 解压Blender
print('\n📂 步骤3/4: 解压Blender...')
if not os.path.exists('/content/blender-4.2.1-linux-x64'):
    !tar -xJf {BLENDER_FILE}
    print('   ✅ 解压完成')
else:
    print('   ✅ Blender已解压')

# 4. 设置环境变量
BLENDER = "/content/blender-4.2.1-linux-x64/blender"
os.environ['BLENDER_PATH'] = BLENDER

# 5. 验证安装
print('\n🔍 步骤4/4: 验证Blender安装...')
!$BLENDER --version

print('\n' + '='*60)
print('✅ Blender 4.2.1 安装完成')
print(f'📍 位置: {BLENDER}')
print('='*60)

🔧 安装Blender 4.2.1

📦 步骤1/4: 安装系统依赖...
   ✅ 系统依赖安装完成

📥 步骤2/4: 下载Blender 4.2.1...
   ✅ Blender压缩包已存在

📂 步骤3/4: 解压Blender...
   ✅ Blender已解压

🔍 步骤4/4: 验证Blender安装...
Blender 4.2.1 LTS
	build date: 2024-08-19
	build time: 23:32:23
	build commit date: 2024-08-19
	build commit time: 11:21
	build hash: 396f546c9d82
	build branch: blender-v4.2-release
	build platform: Linux
	build type: Release
	build c flags:  -Wall -Werror=implicit-function-declaration -Wstrict-prototypes -Werror=return-type -Werror=vla -Wmissing-prototypes -Wno-char-subscripts -Wno-unknown-pragmas -Wpointer-arith -Wunused-parameter -Wwrite-strings -Wlogical-op -Wundef -Winit-self -Wmissing-include-dirs -Wno-div-by-zero -Wtype-limits -Wformat-signedness -Wrestrict -Wno-stringop-overread -Wno-stringop-overflow -Wnonnull -Wabsolute-value -Wuninitialized -Wredundant-decls -Wshadow -Wimplicit-fallthrough=5 -Wno-error=unused-but-set-variable  -march=x86-64-v2 -std=gnu11 -pipe -fPIC -funsigned-char -fno-strict-aliasing -ffp-contr

In [75]:
# 创建GPU配置脚本（基于你的set_cycles_gpu.py）
gpu_script = '''import bpy

# 配置Cycles渲染引擎使用GPU
prefs = bpy.context.preferences.addons["cycles"].preferences

# 尝试OPTIX，如果不支持则使用CUDA
try:
    prefs.compute_device_type = "OPTIX"
    print("✅ 使用OPTIX")
except Exception:
    prefs.compute_device_type = "CUDA"
    print("✅ 使用CUDA")

# 获取所有可用设备
prefs.get_devices()

# 启用所有GPU
gpu_count = 0
for dev in prefs.devices:
    try:
        dev.use = True
        if dev.type in ["CUDA", "OPTIX"]:
            gpu_count += 1
            print(f"   GPU {gpu_count}: {dev.name}")
    except:
        pass

# 设置场景使用GPU
bpy.context.scene.cycles.device = "GPU"

print(f"\\n[Cycles] 计算设备类型: {prefs.compute_device_type}")
print(f"[Cycles] 启用GPU数量: {gpu_count}")
print("✅ GPU渲染配置完成")
'''

# 保存到本地和Drive
with open('/content/set_cycles_gpu.py', 'w') as f:
    f.write(gpu_script)

import shutil
shutil.copy('/content/set_cycles_gpu.py', f'{PROJECT_DIR}/blender_scripts/set_cycles_gpu.py')

print('='*60)
print('✅ GPU配置脚本已创建')
print('='*60)
print('📄 本地: /content/set_cycles_gpu.py')
print(f'💾 备份: {PROJECT_DIR}/blender_scripts/set_cycles_gpu.py')
print('='*60)

✅ GPU配置脚本已创建
📄 本地: /content/set_cycles_gpu.py
💾 备份: /content/drive/MyDrive/LayoutVLM_Project/blender_scripts/set_cycles_gpu.py


In [76]:
import os

os.chdir('/content')

print('='*60)
print('📥 克隆LayoutVLM仓库')
print('='*60)

# 清理旧版本
if os.path.exists('LayoutVLM'):
    print('\n🗑️  清理旧版本...')
    !rm -rf LayoutVLM

# 克隆仓库
print('\n📦 正在克隆...')
!git clone -b colab-compatibility https://github.com/HUMBLEDDDD/LayoutVLM.git

os.chdir('/content/LayoutVLM')

print('\n' + '='*60)
print('✅ LayoutVLM已克隆')
print('='*60)
print(f'📍 位置: {os.getcwd()}')
print('\n📂 项目结构:')
!ls -1

📥 克隆LayoutVLM仓库

🗑️  清理旧版本...

📦 正在克隆...
Cloning into 'LayoutVLM'...
remote: Enumerating objects: 142, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 142 (delta 32), reused 126 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (142/142), 315.73 KiB | 9.87 MiB/s, done.
Resolving deltas: 100% (32/32), done.

✅ LayoutVLM已克隆
📍 位置: /content/LayoutVLM

📂 项目结构:
benchmark_tasks
COLAB_FIX_GUIDE.md
LayoutVLM_Complete.ipynb
main.py
prompts
README.md
requirements.txt
src
test_vision_integration.py
third_party
utils


### 🔄 快速更新代码（可选）

如果你之前已经运行过notebook，只是代码更新了，运行下面的cell快速拉取最新代码：

## 步骤 6️⃣: 安装Python依赖到Blender

📦 **关键步骤**: 将依赖包安装到Blender的Python环境

In [62]:
# 🔍 检查 Blender Python 的 CUDA 支持

import os

BLENDER = os.environ['BLENDER_PATH']

print('='*60)
print('🔍 Blender Python CUDA 诊断')
print('='*60)

# 创建诊断脚本
diag_script = '''import sys
import torch

print("\\n" + "="*60)
print("Blender Python 环境信息")
print("="*60)
print(f"Python 路径: {sys.executable}")
print(f"Python 版本: {sys.version.split()[0]}")

print("\\n" + "="*60)
print("PyTorch CUDA 状态")
print("="*60)
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA 版本: {torch.version.cuda}")
    print(f"GPU 数量: {torch.cuda.device_count()}")
    print(f"当前 GPU: {torch.cuda.get_device_name(0)}")
    print("\\n✅ Blender Python 可以使用 GPU")
else:
    print("\\n⚠️  Blender Python 检测不到 CUDA")
    print("\\n可能原因:")
    print("1. PyTorch 安装的是 CPU 版本")
    print("2. CUDA 库路径配置问题")
    print("3. Blender Python 与系统 CUDA 版本不兼容")

print("="*60)
'''

# 保存诊断脚本
with open('/tmp/check_blender_cuda.py', 'w') as f:
    f.write(diag_script)

# 使用 Blender Python 执行
print('\\n运行 Blender Python 诊断...\\n')
!$BLENDER --background --python /tmp/check_blender_cuda.py

print('\\n' + '='*60)
print('✅ 诊断完成')
print('='*60)

🔍 Blender Python CUDA 诊断
\n运行 Blender Python 诊断...\n
Blender 4.2.1 LTS (hash 396f546c9d82 built 2024-08-19 23:32:23)

Blender Python 环境信息
Python 路径: /content/blender-4.2.1-linux-x64/4.2/python/bin/python3.11
Python 版本: 3.11.7

PyTorch CUDA 状态
PyTorch 版本: 2.9.0+cu128
CUDA 可用: True
CUDA 版本: 12.8
GPU 数量: 1
当前 GPU: NVIDIA A100-SXM4-80GB

✅ Blender Python 可以使用 GPU

Blender quit
\n============================================================
✅ 诊断完成


In [63]:
import os
import torch

print('='*60)
print('⚙️  编译CUDA扩展')
print('='*60)

# 先检查 PyTorch CUDA 状态
print(f'\n📊 当前 PyTorch 状态:')
print(f'   PyTorch 版本: {torch.__version__}')
print(f'   CUDA 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   CUDA 版本: {torch.version.cuda}')
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
else:
    print('   ⚠️  警告: PyTorch 检测不到 CUDA，CUDA 扩展可能无法编译')

os.chdir('/content/LayoutVLM/third_party/Rotated_IoU/cuda_op')

print('\n🔨 开始编译 CUDA 扩展（需要1-2分钟）...')
print('   这会编译 sort_vertices CUDA 模块\n')

# 显示完整输出以便调试
!python setup.py install 2>&1

os.chdir('/content/LayoutVLM')

print('\n' + '='*60)
print('✅ CUDA扩展编译完成')
print('='*60)
print('\n💡 接下来运行诊断 cell 验证是否成功')

⚙️  编译CUDA扩展

📊 当前 PyTorch 状态:
   PyTorch 版本: 2.8.0+cu126
   CUDA 可用: True
   CUDA 版本: 12.6
   GPU: NVIDIA A100-SXM4-80GB

🔨 开始编译 CUDA 扩展（需要1-2分钟）...
   这会编译 sort_vertices CUDA 模块

running install
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:66: EasyInstallDeprecationWarning: easy_install command is deprecated.
!!

        **********************************************************************

## 步骤 8️⃣: 准备数据集

💾 下载Objaverse资产数据集（约2.4GB）

## 步骤 9️⃣: 配置转接API

🔑 **重要**: 填入你的API密钥和配置

## 步骤 🔟: 创建场景配置

📝 定义要生成的3D场景

## 步骤 1️⃣1️⃣: 运行LayoutVLM

🚀 **核心步骤**: 使用Blender Python运行LayoutVLM生成布局

⏱️ 预计时间: 5-15分钟（取决于场景复杂度）

In [69]:
# ========== 快速诊断 ==========

import os
import glob

data_dir = '/content/LayoutVLM/data'

print('='*60)
print('📊 数据集快速诊断')
print('='*60)

# 1. 目录大小
!du -sh {data_dir}

# 2. 文件数量
print('\n文件统计:')
!find {data_dir} -type f | wc -l

# 3. 文件类型分布
print('\n文件类型:')
!find {data_dir} -type f -name "*.*" | sed 's/.*\.//' | sort | uniq -c | sort -rn | head -10

# 4. 目录结构
print('\n目录结构 (前2层):')
!tree {data_dir} -L 2 -d 2>/dev/null || find {data_dir} -maxdepth 2 -type d | head -20

# 5. 查找JSON文件
print('\n数据集中的JSON文件:')
!find {data_dir} -name "*.json" -type f | head -10

📊 数据集快速诊断
3.7G	/content/LayoutVLM/data

文件统计:
6041

文件类型:
   2696 jpg
   1160 png
    674 json
    674 gz
    674 glb
     82 obj
     43 mtl
     38 urdf

目录结构 (前2层):
/content/LayoutVLM/data
/content/LayoutVLM/data/test_asset_dir
/content/LayoutVLM/data/test_asset_dir/e72a40a1adde489b9e0149f94eb7d967
/content/LayoutVLM/data/test_asset_dir/1e835c246ba34627a012f9d6eb2b149d
/content/LayoutVLM/data/test_asset_dir/f6a8d9cf57bb4342840e790d02d1043d
/content/LayoutVLM/data/test_asset_dir/ac4b100dda264a189c837b784a8e242e
/content/LayoutVLM/data/test_asset_dir/2ba80c2563dd4d03a5f719caa0bd1f1c
/content/LayoutVLM/data/test_asset_dir/e57094e36af6477b9b4b1e3307f7a8a8
/content/LayoutVLM/data/test_asset_dir/fc0075723a0a4d7e93aee8642503e171
/content/LayoutVLM/data/test_asset_dir/999f7b6bb4a44a6e933aeb59e68c38ef
/content/LayoutVLM/data/test_asset_dir/7a09f0a44966446183c126358f12b08e
/content/LayoutVLM/data/test_asset_dir/5fdd2647427b4543bd299acff008bc3f
/content/LayoutVLM/data/test_asset_dir/b081286782

## 步骤 1️⃣2️⃣: 查看生成结果

📊 查看生成的布局数据和渲染图片